In [4]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer


In [5]:
movies = pd.read_csv("../data/processed/movies_processed.csv")
print(movies.shape)
movies.head(2)


(4803, 3)


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."


In [7]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
token = os.getenv("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7458.28it/s]


Model loaded!


In [9]:
client = chromadb.PersistentClient(path="../data/chromadb")
collection = client.get_or_create_collection(name="movies")
print("ChromaDB ready!")

ChromaDB ready!


In [10]:
batch_size = 100

for i in range(0, len(movies), batch_size):
    batch = movies.iloc[i : i + batch_size]
    embeddings = model.encode(batch["tags"].tolist()).tolist()

    collection.add(
        ids=[str(x) for x in batch["movie_id"].tolist()],
        embeddings=embeddings,
        documents=batch["tags"].tolist(),
        metadatas=[{"title": t} for t in batch["title"].tolist()],
    )
    print(f"Added {i + len(batch)}/{len(movies)}")

print("Done!")

Added 100/4803
Added 200/4803
Added 300/4803
Added 400/4803
Added 500/4803
Added 600/4803
Added 700/4803
Added 800/4803
Added 900/4803
Added 1000/4803
Added 1100/4803
Added 1200/4803
Added 1300/4803
Added 1400/4803
Added 1500/4803
Added 1600/4803
Added 1700/4803
Added 1800/4803
Added 1900/4803
Added 2000/4803
Added 2100/4803
Added 2200/4803
Added 2300/4803
Added 2400/4803
Added 2500/4803
Added 2600/4803
Added 2700/4803
Added 2800/4803
Added 2900/4803
Added 3000/4803
Added 3100/4803
Added 3200/4803
Added 3300/4803
Added 3400/4803
Added 3500/4803
Added 3600/4803
Added 3700/4803
Added 3800/4803
Added 3900/4803
Added 4000/4803
Added 4100/4803
Added 4200/4803
Added 4300/4803
Added 4400/4803
Added 4500/4803
Added 4600/4803
Added 4700/4803
Added 4800/4803
Added 4803/4803
Done!


In [ ]:
def recommend(movie_title):
    query = movies[movies["title"] == movie_title]["tags"].values[0]
    query_embedding = model.encode(query).tolist()

    results = collection.query(query_embeddings=[query_embedding], n_results=20)

    for movie in results["metadatas"][0]:
        print(movie["title"])


recommend("Avatar")


Avatar
Serenity
Jupiter Ascending
Aliens
Barbarella
Star Trek Beyond
Star Trek Into Darkness
Alien: Resurrection
The Inhabited Island
Gattaca
2001: A Space Odyssey
The Jacket
Battle: Los Angeles
Final Fantasy: The Spirits Within
The Chronicles of Riddick
Prometheus
Independence Daysaster
Home
Red Planet
Star Trek: The Motion Picture
